# P4 · GNN — Drug Candidate Ranking
**Input :** `results/tables/dgi_edges_gnn.csv` (from notebook P3)  
**Outputs:** `models/gcn_best.pt` · `results/tables/gnn_drug_ranking.csv`  
· `results/tables/gnn_node_embeddings.csv` · `results/figures/gnn_*.png`

Trains GCN, GAT, and GraphSAGE on the drug–gene interaction graph and ranks
all drug candidates by the best model's predicted interaction score.

---

### Running on Google Colab with GPU

This notebook is self-contained — it installs its own dependencies and
can be run directly from Colab without cloning the repo first.

1. Upload `results/tables/dgi_edges_gnn.csv` to your Colab session
2. Set `COLAB_MODE = True` in the configuration cell below
3. Runtime → Change runtime type → T4 GPU → Run all

When running locally with the repo cloned, keep `COLAB_MODE = False`.


## Setup — environment detection

In [ ]:
import sys, os
from pathlib import Path

# ── Detect whether we are on Colab or in the local repo ──────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

COLAB_MODE = IN_COLAB   # set to True manually if you uploaded files to Colab
print(f"Running on {'Google Colab' if IN_COLAB else 'local machine'}")
print(f"Colab mode  : {COLAB_MODE}")

## Install dependencies (Colab only)

In [ ]:
if COLAB_MODE:
    import subprocess
    print("Installing PyTorch Geometric...")
    torch_ver = __import__("torch").__version__.split("+")[0]
    # Detect CUDA version for correct wheel
    try:
        import torch; cuda_tag = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"
    except: cuda_tag = "cpu"
    pyg_url = f"https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html"
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "torch-geometric","torch-scatter","torch-sparse","torch-cluster",
                    "--find-links", pyg_url, "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "scikit-learn", "-q"], check=True)
    print("Done.")
else:
    print("Local mode — skipping install (use conda env hcc_drug_discovery)")

## Path configuration

In [ ]:
if COLAB_MODE:
    # ── Colab: load dgi_edges_gnn.csv from the session upload ────────────────
    # Upload the file first via: Files → Upload (left sidebar)
    EDGE_FILE   = Path("dgi_edges_gnn.csv")
    FIGURES_DIR = Path("figures"); FIGURES_DIR.mkdir(exist_ok=True)
    TABLES_DIR  = Path("tables");  TABLES_DIR.mkdir(exist_ok=True)
    MODELS_DIR  = Path("models");  MODELS_DIR.mkdir(exist_ok=True)

    if not EDGE_FILE.exists():
        raise FileNotFoundError(
            "dgi_edges_gnn.csv not found.\n"
            "Upload it via Files → Upload in the Colab sidebar.")
    print(f"Edge file found: {EDGE_FILE}")

else:
    # ── Local repo: auto-detect root via paths.py ─────────────────────────────
    def _find_repo_root(start):
        for p in [start, *start.parents]:
            if (p / "paths.py").exists():
                return p
        raise FileNotFoundError(
            "paths.py not found — run: python scripts/data_download.py")

    REPO_ROOT = _find_repo_root(Path.cwd())
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))

    from paths import PROC_DIR, FIGURES_DIR, TABLES_DIR, MODELS_DIR
    EDGE_FILE = TABLES_DIR / "dgi_edges_gnn.csv"
    print(f"Repo root : {REPO_ROOT}")

print(f"Edge file : {EDGE_FILE}")
print(f"Figures   : {FIGURES_DIR}")
print(f"Models    : {MODELS_DIR}")

## Imports & configuration

In [ ]:
import pickle, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, GATConv, SAGEConv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
warnings.filterwarnings("ignore")

# ── Hyperparameters ───────────────────────────────────────────────────────────
HIDDEN_DIM   = 128
EMBED_DIM    = 64
DROPOUT      = 0.3
LR           = 0.005
WEIGHT_DECAY = 1e-4
N_EPOCHS     = 300
PATIENCE     = 40
GAT_HEADS    = 4
RANDOM_SEED  = 42

DRUG_FEAT_COLS = ["approved","immunotherapy","anti_neoplastic","clinical_phase",
                  "interaction_score","n_publications","source_DGIdb","source_ChEMBL",
                  "source_OpenTargets","type_inhibitor","type_agonist","type_antagonist",
                  "type_antibody","type_binder","type_activator"]
GENE_FEAT_COLS = ["hub_score","survival_target"]
MODEL_COLORS   = {"GCN":"#534AB7","GAT":"#D85A30","GraphSAGE":"#1D9E75"}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {DEVICE}")

## Build drug–gene graph

In [ ]:
edges_df = pd.read_csv(EDGE_FILE)
print(f"Edges loaded: {len(edges_df)}  |  genes: {edges_df.gene.nunique()}  |  drugs: {edges_df.drug.nunique()}")

# ── Node index maps ───────────────────────────────────────────────────────────
all_nodes = pd.concat([edges_df.gene, edges_df.drug]).unique().tolist()
node2idx  = {n: i for i, n in enumerate(all_nodes)}
idx2node  = {i: n for n, i in node2idx.items()}
gene_set  = set(edges_df.gene.unique())
drug_set  = set(edges_df.drug.unique())
n_nodes   = len(all_nodes)

# ── Node feature matrix ───────────────────────────────────────────────────────
all_feat_cols = DRUG_FEAT_COLS + GENE_FEAT_COLS
feat_dim      = len(all_feat_cols)
node_feats    = np.zeros((n_nodes, feat_dim), dtype=np.float32)

gene_rows = edges_df.drop_duplicates("gene").set_index("gene")
for gene, idx in node2idx.items():
    if gene in gene_set and gene in gene_rows.index:
        row = gene_rows.loc[gene]
        for j, col in enumerate(GENE_FEAT_COLS):
            if col in row.index:
                node_feats[idx, len(DRUG_FEAT_COLS)+j] = float(row[col])

drug_rows = edges_df.drop_duplicates("drug").set_index("drug")
for drug, idx in node2idx.items():
    if drug in drug_set and drug in drug_rows.index:
        row = drug_rows.loc[drug]
        for j, col in enumerate(DRUG_FEAT_COLS):
            if col in row.index:
                node_feats[idx, j] = float(row[col])

scaler     = StandardScaler()
node_feats = scaler.fit_transform(node_feats).astype(np.float32)

# ── Edge index & labels ───────────────────────────────────────────────────────
src_n = [node2idx[g] for g in edges_df.gene]
dst_n = [node2idx[d] for d in edges_df.drug]
edge_index = torch.tensor([src_n+dst_n, dst_n+src_n], dtype=torch.long)
labels     = torch.tensor(edges_df.composite_score.values, dtype=torch.float32)
graph_data = Data(x=torch.tensor(node_feats, dtype=torch.float32),
                  edge_index=edge_index).to(DEVICE)

# ── Train/val/test split ──────────────────────────────────────────────────────
indices = list(range(len(edges_df)))
tr_idx, te_idx = train_test_split(indices, test_size=0.15, random_state=RANDOM_SEED)
tr_idx, va_idx = train_test_split(tr_idx,  test_size=0.15/0.85, random_state=RANDOM_SEED)

print(f"Nodes      : {n_nodes}  ({len(gene_set)} genes + {len(drug_set)} drugs)")
print(f"Edges      : {edge_index.shape[1]}  (bidirectional)")
print(f"Feature dim: {feat_dim}")
print(f"Split      : train={len(tr_idx)}  val={len(va_idx)}  test={len(te_idx)}")
print(f"Device     : {DEVICE}")

## Model definitions

In [ ]:
class GCNModel(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN_DIM, out_dim=EMBED_DIM, dropout=DROPOUT):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.bn1   = nn.BatchNorm1d(hidden)
        self.conv2 = GCNConv(hidden, out_dim)
        self.bn2   = nn.BatchNorm1d(out_dim)
        self.head  = nn.Sequential(
            nn.Linear(out_dim*2, out_dim), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(out_dim, 1), nn.Sigmoid())
        self.drop  = dropout
    def encode(self, x, ei):
        x = F.relu(self.bn1(self.conv1(x, ei)))
        x = F.dropout(x, p=self.drop, training=self.training)
        return F.relu(self.bn2(self.conv2(x, ei)))
    def forward(self, x, ei, src, dst):
        z = self.encode(x, ei)
        return self.head(torch.cat([z[src], z[dst]], dim=1)).squeeze(-1)

class GATModel(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN_DIM//GAT_HEADS, out_dim=EMBED_DIM,
                 heads=GAT_HEADS, dropout=DROPOUT):
        super().__init__()
        self.conv1 = GATConv(in_dim, hidden, heads=heads, dropout=dropout, concat=True)
        self.conv2 = GATConv(hidden*heads, out_dim, heads=1, dropout=dropout, concat=False)
        self.bn1   = nn.BatchNorm1d(hidden*heads)
        self.bn2   = nn.BatchNorm1d(out_dim)
        self.head  = nn.Sequential(
            nn.Linear(out_dim*2, out_dim), nn.ELU(),
            nn.Dropout(dropout), nn.Linear(out_dim, 1), nn.Sigmoid())
        self.drop  = dropout
    def encode(self, x, ei):
        x = F.elu(self.bn1(self.conv1(x, ei)))
        x = F.dropout(x, p=self.drop, training=self.training)
        return F.elu(self.bn2(self.conv2(x, ei)))
    def forward(self, x, ei, src, dst):
        z = self.encode(x, ei)
        return self.head(torch.cat([z[src], z[dst]], dim=1)).squeeze(-1)

class SAGEModel(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN_DIM, out_dim=EMBED_DIM, dropout=DROPOUT):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden, aggr="mean")
        self.conv2 = SAGEConv(hidden, out_dim, aggr="mean")
        self.bn1   = nn.BatchNorm1d(hidden)
        self.bn2   = nn.BatchNorm1d(out_dim)
        self.head  = nn.Sequential(
            nn.Linear(out_dim*2, out_dim), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(out_dim, 1), nn.Sigmoid())
        self.drop  = dropout
    def encode(self, x, ei):
        x = F.relu(self.bn1(self.conv1(x, ei)))
        x = F.dropout(x, p=self.drop, training=self.training)
        return F.relu(self.bn2(self.conv2(x, ei)))
    def forward(self, x, ei, src, dst):
        z = self.encode(x, ei)
        return self.head(torch.cat([z[src], z[dst]], dim=1)).squeeze(-1)

def param_count(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f"GCN      : {param_count(GCNModel(feat_dim)):,} parameters")
print(f"GAT      : {param_count(GATModel(feat_dim)):,} parameters")
print(f"GraphSAGE: {param_count(SAGEModel(feat_dim)):,} parameters")

## Training & evaluation functions

In [ ]:
def _et(idx_list):
    """Extract (src, dst, labels) tensors for a given edge index list."""
    src = torch.tensor([node2idx[edges_df.iloc[i].gene] for i in idx_list]).to(DEVICE)
    dst = torch.tensor([node2idx[edges_df.iloc[i].drug] for i in idx_list]).to(DEVICE)
    return src, dst, labels[idx_list].to(DEVICE)

def train_model(model):
    model = model.to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch   = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, "min", patience=15, factor=0.5, min_lr=1e-5)
    s_tr, d_tr, y_tr = _et(tr_idx)
    s_va, d_va, y_va = _et(va_idx)
    best_val, best_state, no_imp = float("inf"), None, 0
    history = {"train_loss":[], "val_loss":[], "lr":[]}

    for ep in range(1, N_EPOCHS+1):
        model.train(); opt.zero_grad()
        loss = F.mse_loss(model(graph_data.x, graph_data.edge_index, s_tr, d_tr), y_tr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        model.eval()
        with torch.no_grad():
            vl = F.mse_loss(
                model(graph_data.x, graph_data.edge_index, s_va, d_va), y_va).item()
        sch.step(vl)
        history["train_loss"].append(loss.item())
        history["val_loss"].append(vl)
        history["lr"].append(opt.param_groups[0]["lr"])
        if vl < best_val:
            best_val   = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_imp     = 0
        else:
            no_imp += 1
        if no_imp >= PATIENCE:
            print(f"    Early stop ep {ep}  best_val={best_val:.4f}"); break
        if ep % 50 == 0:
            print(f"    ep {ep:3d}  train={loss.item():.4f}  val={vl:.4f}")

    model.load_state_dict(best_state)
    return history, best_val

def evaluate(model, idx_list):
    model.eval()
    s, d, yt = _et(idx_list)
    with torch.no_grad():
        yp = model(graph_data.x, graph_data.edge_index, s, d).cpu().numpy()
    yt = yt.cpu().numpy()
    return {"r2"  : float(r2_score(yt, yp)),
            "mse" : float(mean_squared_error(yt, yp)),
            "mae" : float(mean_absolute_error(yt, yp)),
            "pred": yp, "true": yt}

print("Training functions ready")

## Train GCN, GAT, and GraphSAGE

In [ ]:
models_cfg = {
    "GCN"       : GCNModel(feat_dim),
    "GAT"       : GATModel(feat_dim),
    "GraphSAGE" : SAGEModel(feat_dim),
}
all_results = {}

for name, model in models_cfg.items():
    print(f"\n[{name}]")
    history, best_val = train_model(model)
    test_m = evaluate(model, te_idx)
    with torch.no_grad():
        embeds = model.encode(graph_data.x,
                              graph_data.edge_index).cpu().numpy()
    all_results[name] = {
        "history"   : history,
        "best_val"  : best_val,
        "test"      : test_m,
        "embeddings": embeds,
        "model"     : model,
    }
    m = test_m
    print(f"  Test → R²={m['r2']:.4f}  MSE={m['mse']:.4f}  MAE={m['mae']:.4f}")

best_name  = max(all_results, key=lambda n: all_results[n]["test"]["r2"])
best_model = all_results[best_name]["model"]
print(f"\n{'='*50}")
print(f"  Best model : {best_name}")
print(f"  Test R²    : {all_results[best_name]['test']['r2']:.4f}")
print(f"  Test MSE   : {all_results[best_name]['test']['mse']:.4f}")
print(f"{'='*50}")

## Performance figures

In [ ]:
model_names = list(models_cfg.keys())

# ── Training curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(model_names),
                         figsize=(6*len(model_names), 5), facecolor="white")
for ax, name in zip(axes, model_names):
    h   = all_results[name]["history"]
    ep  = range(1, len(h["train_loss"])+1)
    col = MODEL_COLORS[name]
    m   = all_results[name]["test"]
    ax.plot(ep, h["train_loss"], color=col, lw=2, label="Train")
    ax.plot(ep, h["val_loss"],   color=col, lw=1.5, ls="--", alpha=0.65, label="Val")
    ax.fill_between(ep, h["train_loss"], h["val_loss"], color=col, alpha=0.06)
    ax.axvline(int(np.argmin(h["val_loss"]))+1, color=col, lw=0.8, ls=":", alpha=0.6)
    ax.set_title(f"{name}\nR²={m['r2']:.4f}  MSE={m['mse']:.4f}", fontsize=11,
                 color=col if name==best_name else "black",
                 fontweight="bold" if name==best_name else "normal")
    ax.set_xlabel("Epoch"); ax.set_ylabel("MSE Loss")
    ax.legend(fontsize=8); ax.spines[["top","right"]].set_visible(False)
    if name == best_name: ax.set_facecolor("#f8f6ff")
fig.suptitle(f"GNN training curves — best: {best_name}", fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "gnn_training_curves.png", dpi=200, bbox_inches="tight")
plt.show()

# ── Model comparison ──────────────────────────────────────────────────────────
metric_defs = [("r2","R² (↑)",True),("mse","MSE (↓)",False),("mae","MAE (↓)",False)]
fig, axes   = plt.subplots(1, 3, figsize=(13, 5), facecolor="white")
for ax, (metric, label, higher) in zip(axes, metric_defs):
    vals   = [all_results[n]["test"][metric] for n in model_names]
    colors = [MODEL_COLORS[n] for n in model_names]
    best_v = max(vals) if higher else min(vals)
    bars   = ax.bar(model_names, vals, color=colors, width=0.5)
    for bar, val in zip(bars, vals):
        bar.set_alpha(1.0 if val == best_v else 0.55)
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(vals)*0.015,
                f"{val:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
    ax.set_title(label, fontsize=11); ax.set_ylim(0, max(vals)*1.22)
    ax.spines[["top","right"]].set_visible(False)
fig.suptitle("Model comparison — test set", fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "gnn_model_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

# ── Scatter: predicted vs actual ──────────────────────────────────────────────
fig, axes = plt.subplots(1, len(model_names), figsize=(5*len(model_names), 5), facecolor="white")
for ax, name in zip(axes, model_names):
    m   = all_results[name]["test"]
    col = MODEL_COLORS[name]
    ax.scatter(m["true"], m["pred"], color=col, s=90, alpha=0.85,
               edgecolors="white", linewidths=0.6, zorder=3)
    lo = min(m["true"].min(), m["pred"].min())-0.05
    hi = max(m["true"].max(), m["pred"].max())+0.05
    ax.plot([lo,hi],[lo,hi],"k--",lw=1,alpha=0.45,zorder=2)
    for t, p in zip(m["true"], m["pred"]):
        ax.plot([t,t],[t,p],color=col,lw=0.6,alpha=0.30,zorder=1)
    ax.set_xlim(lo,hi); ax.set_ylim(lo,hi)
    ax.set_xlabel("True score"); ax.set_ylabel("Predicted score")
    ax.set_title(f"{name}  R²={m['r2']:.4f}", fontsize=10,
                 color=col if name==best_name else "black")
    ax.spines[["top","right"]].set_visible(False)
    if name == best_name: ax.set_facecolor("#f8f6ff")
fig.suptitle("Predicted vs actual — test set", fontsize=12, y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "gnn_predicted_vs_actual.png", dpi=200, bbox_inches="tight")
plt.show()

## Drug candidate ranking

In [ ]:
best_model.eval()
all_src = torch.tensor([node2idx[g] for g in edges_df.gene]).to(DEVICE)
all_dst = torch.tensor([node2idx[d] for d in edges_df.drug]).to(DEVICE)
with torch.no_grad():
    gnn_scores = best_model(
        graph_data.x, graph_data.edge_index, all_src, all_dst).cpu().numpy()

ranking = edges_df.copy()
ranking["gnn_score"]      = gnn_scores.round(4)
ranking["original_score"] = ranking.composite_score
ranking["score_delta"]    = (ranking.gnn_score - ranking.original_score).round(4)
ranking = ranking.sort_values("gnn_score", ascending=False).reset_index(drop=True)
ranking["rank"] = ranking.index + 1

# ── Ranking chart ─────────────────────────────────────────────────────────────
PHASE_COLORS = {0:"#D3D1C7",1:"#B5D4F4",2:"#378ADD",3:"#185FA5",4:"#1D9E75"}
top_n    = min(25, len(ranking))
top_df   = ranking.head(top_n).iloc[::-1].reset_index(drop=True)
bar_cols = [PHASE_COLORS.get(int(p),"#888780") for p in top_df.clinical_phase]

fig, ax = plt.subplots(figsize=(14, max(8, top_n*0.38)), facecolor="white")
ax.barh(range(len(top_df)), top_df.gnn_score, color=bar_cols, alpha=0.88, height=0.72)
for i, row in top_df.iterrows():
    ax.text(row.gnn_score+0.005, i,
            "★" if row.approved else "○",
            va="center", fontsize=11,
            color="#1D9E75" if row.approved else "#888780")
    if "original_score" in row:
        ax.scatter(row.original_score, i, marker="|", s=60,
                   color="#444441", zorder=5, linewidths=1.5)

labels = [row.drug + "   →   " + row.gene for _, row in top_df.iterrows()]
ax.set_yticks(range(len(top_df))); ax.set_yticklabels(labels, fontsize=8.5)
ax.set_xlabel(f"GNN-predicted interaction score ({best_name})", fontsize=11)
ax.set_title(f"Top {top_n} drug candidates
"
             "★ = approved   ○ = not approved   | = original score",
             fontsize=11, pad=12)

import matplotlib.patches as mp
legend_p = [mp.Patch(color=c, label=f"Phase {k}" if k>0 else "Preclinical", alpha=0.88)
            for k, c in sorted(PHASE_COLORS.items())]
ax.legend(handles=legend_p, loc="lower right", fontsize=8.5,
          framealpha=0.85, title="Clinical phase")
ax.spines[["top","right"]].set_visible(False)
ax.set_xlim(0, 1.10); ax.axvline(0.5, color="#ccc", lw=0.8, ls="--")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "gnn_drug_ranking.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"\nTop 10 drug candidates ({best_name}):")
ranking[["rank","drug","gene","gnn_score","approved","clinical_phase"]].head(10)

## Save model, embeddings & ranking

In [ ]:
# Best model weights
torch.save(best_model.state_dict(), MODELS_DIR / "gcn_best.pt")

# Feature scaler (needed to use the model at inference time)
with open(MODELS_DIR / "feature_scaler.pkl","wb") as f:
    pickle.dump(scaler, f)

# Node embeddings (64-dim learned representations)
embed_df = pd.DataFrame(
    all_results[best_name]["embeddings"],
    columns=[f"dim_{i}" for i in range(EMBED_DIM)])
embed_df.insert(0, "node",      [idx2node[i] for i in range(n_nodes)])
embed_df.insert(1, "node_type", ["gene" if embed_df.iloc[i].node in gene_set
                                  else "drug" for i in range(n_nodes)])
embed_df.to_csv(TABLES_DIR / "gnn_node_embeddings.csv", index=False)

# Full drug ranking
rank_cols = ["rank","gene","drug","gnn_score","original_score","score_delta",
             "approved","clinical_phase","interaction_type","source"]
ranking[[c for c in rank_cols if c in ranking.columns]].to_csv(
    TABLES_DIR / "gnn_drug_ranking.csv", index=False)

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "="*55)
print("  GNN pipeline complete")
print("="*55)
for name in model_names:
    m   = all_results[name]["test"]
    tag = "  ← best" if name == best_name else ""
    print(f"  {name:<12}  R²={m['r2']:.4f}  MSE={m['mse']:.4f}  MAE={m['mae']:.4f}{tag}")

print()
outputs = [
    (MODELS_DIR / "gcn_best.pt",                   "Model weights"),
    (MODELS_DIR / "feature_scaler.pkl",            "Feature scaler"),
    (TABLES_DIR / "gnn_drug_ranking.csv",          "Drug ranking"),
    (TABLES_DIR / "gnn_node_embeddings.csv",       "Node embeddings"),
    (FIGURES_DIR / "gnn_training_curves.png",      "Training curves"),
    (FIGURES_DIR / "gnn_model_comparison.png",     "Model comparison"),
    (FIGURES_DIR / "gnn_drug_ranking.png",         "Drug ranking chart"),
]
for path, desc in outputs:
    status = "✓" if path.exists() else "✗ MISSING"
    size   = f"{path.stat().st_size//1024} KB" if path.exists() else ""
    print(f"  {status}  {desc:<25} {size}")

print()
print(f"Top 5 approved drug candidates:")
print(ranking[ranking.approved][["rank","drug","gene","gnn_score"]].head(5).to_string(index=False))